# M3 · RAG jako narzędzie: raporty, chunking, AI Search

> VP of Sales, TechRetail Corp: *"Mój zespół chce też chatbota, który odpowiada na pytania kontekstowe, nie tylko liczbowe. Czy asystent może czytać nasze raporty i odpowiadać z cytatami?"*

Tabela odpowiada "ile", a raport odpowiada "dlaczego" i "co z tym zrobić". W tym module 10 raportów PDF staje się drugim źródłem wiedzy agenta.

```
OFFLINE (raz)   PDF w Volume → ai_parse_document → tekst per strona → chunki 600/100 → embeddingi → indeks AI Search
ONLINE (pytanie)  pytanie → embedding → top-3 chunki → prompt z fragmentami → odpowiedź z cytatami [raport #fragment]
```

| Część | Co robisz | Lab |
|---|---|---|
| 1 | parsowanie PDF (albo gotowy checkpoint) | wywołanie `ai_parse_document` |
| 2 | chunking i jego parametry | `chunk_size`, `chunk_overlap`, separatory |
| 3 | embedding policzony ręcznie | podobieństwo kosinusowe |
| 4 | indeks AI Search i RAG z cytatami | kontekst i prompt |
| 5 | ANN, HYBRID, FULL_TEXT, filtr; Playground z indeksem | UI |

**Wzorzec na innych danych (zrzuty w prezentacji):** `workshop/pattern/p3_rag_robotics`, czyli cały RAG na raportach o robotyce (3 strony, chunking 2000/200). U nas raporty mają 5 stron i 600/100. Zapamiętaj, skąd bierze się ta różnica; notebook możesz przejść sam po warsztacie.

**AI Search** to nowa nazwa **Vector Search** (od czerwca 2026). Klasy w bibliotekach, np. `VectorSearchRetrieverTool`, jeszcze noszą starą nazwę.

**Free Edition:** masz 1 endpoint AI Search. Jeśli nie jest gotowy (`PROVISIONING`), notebook sam przejdzie w tryb offline: `retrieve_local()` liczy podobieństwo na przygotowanych embeddingach. Ćwiczenia działają w obu trybach.

## Mapa ścieżek

Ścieżka A to pełny cel modułu. B i C robisz, gdy skończysz A.

| Ścieżka | Co robisz | Gotowe, gdy | Sekcja |
|---|---|---|---|
| **A · Razem** (TechRetail) | od PDF do RAG z cytatami na 10 raportach; sam ustawiasz parametry chunkingu i piszesz prompt z cytatami | `custom_rag` cytuje `[raport #fragment]`, a na pytanie spoza raportów odpowiada "Nie ma tego w raportach" | części 1-6 |
| **B · Samodzielnie** (Bakehouse) | RAG na opiniach klientów sieci piekarni, z progiem trafności | pytanie o opinie dostaje odpowiedź z cytatami `[opinia ..., franczyza ...]`, a pytanie spoza opinii "Nie ma tego w opiniach." bez wywołania modelu | "B · Samodzielnie: RAG na opiniach klientów Bakehouse, z progiem trafności" |
| **C · Wyzwanie** (TechRetail) | pomiar, czy rozmiar fragmentu zmienia trafność wyszukiwania | tabela 3 wariantów chunkingu: liczba fragmentów, hit@1, hit@3 | "C · Wyzwanie: czy chunking naprawdę zmienia trafność?" |


In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
dbutils.library.restartPython()

**Infrastruktura.** Konfiguracja wspólna dla wszystkich modułów. Uruchom i czytaj dalej, tu nie ma nic do nauczenia.


In [ ]:
# Wspólna konfiguracja warsztatu. Ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
BH_SCHEMA = "bakehouse"       # ścieżka B: kopie danych Bakehouse i funkcje-narzędzia
AIRBNB_SCHEMA = "airbnb"      # ścieżka C: oferty Airbnb i funkcje-narzędzia
POLICY_SCHEMA = "governance"  # maski i filtry ścieżek B i C: poza schematami, które MCP wystawia agentowi
BH_TRANSACTIONS = f"{CATALOG}.{BH_SCHEMA}.transactions"
BH_REVIEWS = f"{CATALOG}.{BH_SCHEMA}.reviews"
AIRBNB_TABLE = f"{CATALOG}.{AIRBNB_SCHEMA}.listings"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

import logging
# MLflow w notebooku serverless (UI) wypisuje przy tracingu stos Py4JSecurityException z "resolving tags".
# To ostrzeżenie, nie błąd. Trace zapisuje się poprawnie, a wyciszamy je, żeby nikt nie wziął go za błąd.
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

**Infrastruktura.** Komórka przygotowuje moduł. Uruchom ją i czytaj dalej.

Tworzy połączenie z Databricks (`w`) i klienta modelu (`llm`), zapamiętuje Twój login i ścieżkę do danych w repozytorium. Najważniejsza jest flaga `RUN_PARSE`. Przy `False` (domyślnie) notebook bierze gotowy wynik parsowania PDF z pliku. Przy `True` parsuje PDF od nowa, co trwa dłużej. Na końcu komórka sprawdza, czy `00_setup` założył tabelę z fragmentami raportów.


In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F

w = WorkspaceClient()
llm = w.serving_endpoints.get_open_ai_client()
USERNAME = spark.sql("SELECT current_user()").first()[0]
DATA_DIR = Path(os.getcwd()).parent / "data"
SEARCH_COLUMNS = ["chunk_id", "content", "doc_id", "filename", "chunk_position"]

# True: parsujemy 10 PDF z Volume (kilka minut). False: bierzemy gotowy checkpoint.
RUN_PARSE = False
PARSED_PAGES_PATH = f"{VOLUME_PATH}/parsed_pages"  # obrazy stron zapisywane przez parser

assert spark.catalog.tableExists(CHUNKS_TABLE), "Brak tabeli chunków: uruchom najpierw 00_setup."
print(f"Użytkownik: {USERNAME} | dane: {DATA_DIR} | RUN_PARSE={RUN_PARSE}")


**Infrastruktura.** Klient embeddingów i funkcja `embed()`. Uruchom i czytaj dalej.

`embed(texts)` dostaje listę tekstów i zwraca po jednym wektorze na każdy tekst. Wektor to 1024 liczby, które opisują znaczenie tekstu. Z tej funkcji korzysta część 3 i wyszukiwanie bez indeksu.

Model embeddingów ma limit zapytań na cały workspace. Na Free Edition to Twój własny workspace, na sali Premium jeden wspólny dla wszystkich. Gdy limit się wyczerpie (błąd `429`), klient sam czeka i ponawia zapytanie, zamiast przerywać notebook.


In [ ]:
# Klient embeddingów: ten sam protokół OpenAI co w M1, tylko model zamienia tekst na wektor.
# Limit zapytań jest wspólny dla całego workspace, a na sali dzieli go cała grupa,
# więc max_retries sprawia, że klient sam odczekuje i ponawia zamiast przerywać notebook.
embedding_client = llm.with_options(max_retries=5, timeout=30)


def embed(texts: list) -> np.ndarray:
    """Zamienia listę tekstów na wektory. Limit 429 zależy od rozmiaru żądania."""
    response = embedding_client.embeddings.create(model=EMBEDDING_ENDPOINT, input=texts)
    return np.array([row.embedding for row in response.data])


**Infrastruktura.** Zamawia endpoint i indeks AI Search i nie czeka, aż będą gotowe. Indeks buduje się w tle, gdy Ty robisz części 1-3. W części 4 sprawdzimy tylko, czy jest gotowy. Uruchom i czytaj dalej.

- Endpoint to serwer, na którym działa wyszukiwanie. Komórka zakłada go, jeśli jeszcze nie istnieje.
- `create_chunks_index()` zakłada indeks nad tabelą fragmentów `retail_rag_chunks`. To indeks typu Delta Sync, czyli sam czyta dane z tabeli. Wektory dla kolumny `content` liczy Databricks, tym samym modelem embeddingów, którego używa notebook. `TRIGGERED` oznacza, że indeks pobiera zmiany z tabeli dopiero wtedy, gdy o to poprosisz.
- Indeks da się założyć tylko na działającym endpoincie. Jeśli endpoint jeszcze wstaje, indeks założy komórka w części 4.


In [ ]:
# Zamawiamy indeks i nie czekamy na niego: buduje się w tle, gdy robimy części 1-3.
from databricks.ai_search.client import AISearchClient

search_client = AISearchClient(disable_notice=True)


def create_chunks_index() -> None:
    """Indeks Delta Sync nad tabelą chunków. Embeddingi kolumny `content` liczy Databricks."""
    search_client.create_delta_sync_index(
        endpoint_name=SEARCH_ENDPOINT,
        index_name=SEARCH_INDEX,
        primary_key="chunk_id",
        source_table_name=CHUNKS_TABLE,
        pipeline_type="TRIGGERED",
        embedding_source_column="content",
        embedding_model_endpoint_name=EMBEDDING_ENDPOINT,
        columns_to_sync=["doc_id", "filename", "chunk_position"],
    )


if not search_client.endpoint_exists(SEARCH_ENDPOINT):
    search_client.create_endpoint(name=SEARCH_ENDPOINT, endpoint_type="STANDARD")

# Indeks da się założyć dopiero na działającym endpoincie; jeśli ten wstaje, zrobi to część 4.
endpoint_state = search_client.get_endpoint(SEARCH_ENDPOINT)["endpoint_status"]["state"]
if endpoint_state == "ONLINE" and not search_client.index_exists(SEARCH_ENDPOINT, SEARCH_INDEX):
    create_chunks_index()

print(f"Endpoint {SEARCH_ENDPOINT}: {endpoint_state}. Indeks odbierzemy w części 4.")


## 1. Od PDF do tekstu: `ai_parse_document`

> **Cel:** zobaczyć, że PDF staje się tekstem razem z tabelami i opisem wykresu.
> **Gotowe, gdy:** w wyniku widzisz treść strony, a nie ścieżkę do pliku.


Analitycy TechRetail przygotowali wcześniej 10 raportów, które leżą w Volume `retail_docs` (skopiował je `00_setup`). `ai_parse_document` to funkcja SQL, która z PDF wyciąga strony, elementy (tytuły, tekst, tabele, wykresy) i ich położenie na stronie (bbox).

**ZADANIE 6 wykonuje się tylko przy `RUN_PARSE = True`.** Flagę ustawia komórka `m3-context` wyżej, nie komórka zadania. Przy domyślnym `False` notebook bierze gotowy tekst z checkpointu, a Twój kod z zadania nie zostanie uruchomiony.

Parsowanie 10 plików trwa kilka minut i na Free Edition potrafi trafić na limit (sama funkcja, z obrazami stron i opisami wykresów, działała na Free w testach z lipca 2026). Dlatego domyślnie (`RUN_PARSE = False`) wczytujemy wynik, który prowadzący zapisał tą samą komórką na workspace Premium.

**Opcjonalnie:** uzupełnij wywołanie funkcji w gałęzi `RUN_PARSE`. Jeśli masz czas i kredyt, ustaw `RUN_PARSE = True` w komórce wyżej i porównaj liczbę znaków z checkpointem.

In [ ]:
if RUN_PARSE:
    docs_df = spark.sql(f"""
        WITH parsed_docs AS (
            SELECT
                _metadata.file_name AS filename,
                ai_parse_document(
                    content,
                    MAP('version', '2.0', 'imageOutputPath', '{PARSED_PAGES_PATH}',
                        'descriptionElementTypes', '*')
                ) AS parsed
            FROM READ_FILES('{VOLUME_PATH}/', format => 'binaryFile')
            WHERE _metadata.file_name LIKE '%.pdf'
        )
        SELECT
            REPLACE(filename, '.pdf', '') AS doc_id,
            filename,
            concat_ws('\\n\\n', transform(
                try_cast(parsed:document:elements AS ARRAY<VARIANT>),
                element -> try_cast(element:content AS STRING)
            )) AS content,
            to_json(parsed) AS parsed_json
        FROM parsed_docs
        WHERE is_variant_null(parsed:error_status)
    """)
    print("Parsowanie na żywo: ai_parse_document")
else:
    checkpoint = DATA_DIR / "checkpoints" / "retail_rag_docs.parquet"
    docs_df = spark.createDataFrame(pd.read_parquet(checkpoint))
    print(f"Tryb offline: tekst z przygotowanego pliku {checkpoint.name}")


In [ ]:
docs_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(DOCS_TABLE)

columns = ["doc_id", "filename", F.length("content").alias("znaków")]
display(spark.table(DOCS_TABLE).select(*columns).orderBy("doc_id"))


In [ ]:
# Co zwrócił parser: elementy i ich typy. Wymaga RUN_PARSE = True, bo checkpoint
# nie ma kolumny parsed_json.
if "parsed_json" not in spark.table(DOCS_TABLE).columns:
    print("Pominięte: metadane parsowania są tylko po RUN_PARSE = True.")
else:
    display(spark.sql(f"""
        SELECT value:type::string AS typ_elementu, COUNT(*) AS n
        FROM (SELECT parse_json(parsed_json) AS parsed FROM {DOCS_TABLE}),
             LATERAL variant_explode(parsed:document:elements)
        GROUP BY 1
        ORDER BY n DESC
    """))


**Składniki podglądu.** Ta komórka tylko przygotowuje klocki do rysowania i sama nic jeszcze nie pokazuje. Podgląd uruchamia następna komórka.

- `TYPE_COLORS` przypisuje kolor każdemu rodzajowi elementu, który rozpoznał parser: tytuł, nagłówek sekcji, akapit, tabela, rysunek, podpis, nagłówek i stopka strony.
- `ElementBox` to jeden element strony w prostej formie: rodzaj, rozpoznana treść i cztery współrzędne ramki (lewo, góra, prawo, dół).
- `boxes_on_page(document, page_index)` przegląda wszystkie elementy z wyniku `ai_parse_document`, zostawia te z jednej strony i zamienia każdy na `ElementBox`. Współrzędne bierze z pola `bbox`, które parser podaje przy każdym elemencie. Strony liczy od zera.
- `svg_rectangle(box)` zamienia jeden `ElementBox` na ramkę w formacie SVG: kolor według rodzaju, lekkie wypełnienie i podpowiedź z treścią elementu, która pojawia się po najechaniu myszą.

Nie musisz wchodzić w szczegóły SVG. Ważne jest to, że parser zna położenie każdego elementu na stronie, a nie tylko jego tekst.


In [ ]:
# Składniki podglądu: kolory typów, wyciąganie ramek i rysowanie jednej ramki.
import base64
import html
import json
from collections import Counter
from typing import NamedTuple

TYPE_COLORS = {"title": "#7c3aed", "section_header": "#2563eb", "text": "#16a34a",
               "table": "#ea580c", "figure": "#db2777", "caption": "#0891b2",
               "page_header": "#6b7280", "page_footer": "#6b7280"}


class ElementBox(NamedTuple):
    kind: str
    content: str
    left: float
    top: float
    right: float
    bottom: float


def boxes_on_page(document: dict, page_index: int) -> list[ElementBox]:
    """Elementy jednej strony. Parser podaje ich współrzędne w polu bbox."""
    boxes = []
    for element in document.get("elements", []):
        first_box = (element.get("bbox") or [{}])[0]
        coordinates = first_box.get("coord") or []
        if first_box.get("page_id") != page_index or len(coordinates) < 4:
            continue
        content = element.get("content") or element.get("description") or ""
        boxes.append(ElementBox(str(element.get("type")), str(content), *coordinates[:4]))
    return boxes


def svg_rectangle(box: ElementBox) -> str:
    """Jedna ramka. Najechanie myszą pokazuje rozpoznaną treść elementu."""
    color = TYPE_COLORS.get(box.kind, "#64748b")
    tooltip = html.escape(f"{box.kind}: {box.content[:300]}")
    return (f'<g><title>{tooltip}</title><rect '
            f'x="{min(box.left, box.right)}" y="{min(box.top, box.bottom)}" '
            f'width="{abs(box.right - box.left)}" height="{abs(box.bottom - box.top)}" '
            f'fill="{color}" fill-opacity="0.12" stroke="{color}" stroke-width="3"/></g>')


**Podgląd dwóch stron.** Tu z klocków wyżej powstaje obraz strony z ramkami, a na końcu komórka wywołuje go dla pierwszego raportu.

- `render_parsed_page(parsed_json, page_no=1)` bierze wynik parsera dla jednego dokumentu i numer strony (liczony od 1). Odszukuje obraz tej strony, który parser zapisał na dysku, nakłada na niego ramki wszystkich rozpoznanych elementów i wyświetla całość razem z legendą, ile elementów którego rodzaju jest na stronie. Nic nie zwraca. Gdy obrazu strony nie ma, wypisuje komunikat z jego ścieżką i kończy.

Komórka pokazuje strony 1 i 2 pierwszego raportu. Najedź myszą na ramkę, żeby zobaczyć rozpoznaną treść elementu. Podgląd działa tylko po parsowaniu na żywo (`RUN_PARSE = True`), bo checkpoint nie ma kolumny `parsed_json`. W innym wypadku komórka wypisze, że pomija podgląd.


In [ ]:
def render_parsed_page(parsed_json: str, page_no: int = 1) -> None:
    """Obraz strony PDF z ramkami elementów rozpoznanych przez parser."""
    document = json.loads(parsed_json).get("document", {})
    pages = document.get("pages", [])
    page = next((p for p in pages if int(p.get("id", -1)) == page_no - 1), None)
    image_path = (page or {}).get("image_uri") or ""
    if not os.path.isfile(image_path):
        print(f"Brak obrazu strony {page_no}: {image_path}")
        return

    boxes = boxes_on_page(document, page_no - 1)
    width = max((max(box.left, box.right) for box in boxes), default=1000)
    height = max((max(box.top, box.bottom) for box in boxes), default=1400)
    shapes = "".join(svg_rectangle(box) for box in boxes)
    counts = sorted(Counter(box.kind for box in boxes).items())
    legend = ", ".join(f"{kind}: {n}" for kind, n in counts)
    image_b64 = base64.b64encode(Path(image_path).read_bytes()).decode("ascii")

    displayHTML(
        f'<div style="max-width:720px;font-family:Arial">'
        f'<p><b>Strona {page_no}</b> · {legend} · najedź na ramkę, żeby zobaczyć treść</p>'
        f'<div style="position:relative">'
        f'<img src="data:image/png;base64,{image_b64}" style="width:100%;display:block">'
        f'<svg viewBox="0 0 {width} {height}" preserveAspectRatio="none" '
        f'style="position:absolute;inset:0;width:100%;height:100%">{shapes}</svg></div></div>')


if "parsed_json" not in spark.table(DOCS_TABLE).columns:
    print("Pominięte: podgląd ramek wymaga RUN_PARSE = True.")
else:
    first_report = spark.table(DOCS_TABLE).orderBy("doc_id").first()
    print(first_report["filename"])
    for page_number in (1, 2):
        render_parsed_page(first_report["parsed_json"], page_number)


## 2. Chunking: dlaczego nie cały dokument

> **Cel:** dobrać rozmiar fragmentu do swoich dokumentów, zamiast przepisać 600/100 z tutoriala.
> **Gotowe, gdy:** masz trzy warianty obok siebie i potrafisz powiedzieć, który wybierasz i dlaczego.


Gdyby indeks miał 10 wierszy (10 całych raportów):
- embedding pięciostronicowego raportu to "średnia" całej treści, więc pytanie o retencję VIP pasuje równie słabo do każdego dokumentu;
- do promptu trafia cały raport: za dużo szumu albo tekst ucięty w połowie tabeli;
- na 10 wierszach filtry i wyszukiwanie hybrydowe nie mają czego przeszukiwać.

Dlatego tniemy tekst na fragmenty po ok. **600 znaków z nakładką 100**. Separator `== page ==` stoi na liście pierwszy, więc splitter najchętniej tnie na granicy strony. Nasze raporty mają 5 stron, co daje 5-8 fragmentów na raport. Przy `chunk_size=2000` wyszłyby po 2, za mało, żeby wyszukiwanie miało z czego wybierać. Od góry ogranicza nas okno modelu embeddingowego (rzędu 512 tokenów): nadmiar jest ucinany bez ostrzeżenia.

**Rodzaje chunkingu.** Nasz splitter (`RecursiveCharacterTextSplitter`) to wariant rekurencyjny; warto znać pozostałe, bo wybór zależy od dokumentów, nie od biblioteki.

| Rodzaj | Jak tnie | Kiedy | U nas |
|---|---|---|---|
| **Stały rozmiar** | co N znaków lub tokenów, z nakładką | logi, transkrypcje, tekst bez struktury | punkt odniesienia w `m3-chunking-lab` (2000/0) |
| **Rekurencyjny** | próbuje separatorów po kolei (strona, akapit, zdanie, słowo), aż fragment zmieści się w limicie | większość dokumentów z akapitami; domyślny wybór | **tak**, 600/100, separatory z `== page ==` na czele |
| **Po strukturze dokumentu** | granice to nagłówki, strony, komórki tabeli, sekcje Markdown/HTML | raporty, dokumentacja, umowy, gdy sekcja jest naturalną jednostką sensu | częściowo: `== page ==` jako pierwszy separator; `ai_parse_document` daje elementy (nagłówki, tabele), z których da się ciąć po sekcjach |
| **Zdaniowy / semantyczny** | dzieli na zdania, a potem skleja sąsiednie, dopóki są o tym samym (spadek podobieństwa embeddingów = granica) | teksty bez nagłówków, w których tematy zmieniają się w środku akapitu | nie; kosztuje embedding każdego zdania |
| **Bez chunkingu** | jeden dokument = jeden wektor | krótkie teksty: opinie, tickety, notatki | **tak**, ścieżka B (opinie Bakehouse) |
| **Mały fragment, duży kontekst** (parent-child) | szuka po małych fragmentach, a do promptu wkłada cały akapit lub sekcję nadrzędną | gdy trafność wymaga precyzji, a odpowiedź kontekstu | nie; w M3 do promptu trafia ten sam fragment, który został znaleziony |

Reguła praktyczna z karty wzorca: fragment ma być tak długi, jak typowa **jednostka sensu** w Twoich dokumentach, i mieścić się w oknie modelu embeddingowego. Ścieżka C mierzy, czy ta intuicja trzyma się na naszych raportach.

Komórka poniżej zamienia wynik parsera na **tekst per strona** i zapisuje go w `markdown_df`. Z tej zmiennej korzysta chunking niżej oraz ścieżka C, dlatego ta komórka nie jest opcjonalna.

Kod jest w dwóch komórkach:

- Pierwsza definiuje dwie funkcje. `parsed_json_to_plain_text()` bierze wynik parsera, czyli JSON z elementami strony (tytuły, akapity, tabele), i składa z niego zwykły tekst, strona po stronie. Strony oddziela znacznikiem `== page ==`, w którym chunking najchętniej tnie. `html_to_plain_text()` pomaga przy tabelach. Parser zwraca je jako HTML, a ta funkcja usuwa znaczniki i zostawia sam tekst.
- Druga bierze wynik parsowania z tabeli z części 1, a gdy część 1 była pominięta, z gotowego pliku. Jeśli tekstu jeszcze nie ma, uruchamia funkcję na każdym dokumencie. Na końcu wypisuje początek pierwszego raportu.


In [ ]:
# Wynik parsera to elementy z całego dokumentu. Tu składamy z nich tekst strona po stronie.
import html
import json
import re

from pyspark.sql.types import StringType


def html_to_plain_text(value: str) -> str:
    """Tabele przychodzą jako HTML: usuwamy tagi, zachowując podziały wierszy."""
    with_breaks = re.sub(r"</(p|div|tr|li|h[1-6])>", "\n", value, flags=re.IGNORECASE)
    no_tags = re.sub(r"<[^>]+>", " ", with_breaks)
    return re.sub(r"[ \t]+", " ", html.unescape(no_tags)).strip()


def parsed_json_to_plain_text(parsed_json: str) -> str:
    """Spłaszcza elementy do tekstu per strona; strony rozdziela == page ==."""
    document = json.loads(parsed_json).get("document") or {}
    pages = document.get("pages") or []
    text_by_page = {int(page.get("id", index)): [] for index, page in enumerate(pages)}

    for element in sorted(document.get("elements") or [], key=lambda e: e.get("id", 0)):
        content = element.get("content") or element.get("description")
        text = html_to_plain_text(str(content)) if content else ""
        if not text:
            continue
        bbox = element.get("bbox") or []
        page_id = int(bbox[0].get("page_id", 0)) if bbox else 0
        text_by_page.setdefault(page_id, []).append(text)

    order = [int(page.get("id", index)) for index, page in enumerate(pages)] or sorted(text_by_page)
    return "\n== page ==\n".join("\n".join(text_by_page.get(page_id, [])) for page_id in order)


In [ ]:
# Część 1 jest opcjonalna i tylko ona zapisuje DOCS_TABLE. Bez niej bierzemy ten sam checkpoint.
if spark.catalog.tableExists(DOCS_TABLE):
    docs = spark.table(DOCS_TABLE)
else:
    checkpoint = DATA_DIR / "checkpoints" / "retail_rag_docs.parquet"
    print(f"Brak {DOCS_TABLE} (część 1 pominięta): wczytuję {checkpoint.name}")
    docs = spark.createDataFrame(pd.read_parquet(checkpoint))

# markdown_df: tekst per strona w kolumnie plain_text. Korzysta z niej chunking niżej i ścieżka C.
if "plain_text" in docs.columns:
    markdown_df = docs
else:
    to_plain_text = F.udf(parsed_json_to_plain_text, StringType())
    markdown_df = docs.withColumn("plain_text", to_plain_text(F.col("parsed_json")))

print(markdown_df.orderBy("doc_id").first()["plain_text"][:1200])


**Lab:** ustaw parametry splittera i sprawdź, ile fragmentów powstaje. Spróbuj trzech wariantów: **300/50, 600/100 i 1500/150**, czyli tych samych, które mierzy ścieżka C, więc będziesz mógł porównać liczby (u nas: 120, 57 i 22 fragmenty). Dla odniesienia sprawdź też 2000/0, czyli duże fragmenty bez nakładki. Zapisz, jak zmienia się liczba i średnia długość fragmentów.

Twój eksperyment zostaje w pamięci notebooka. Indeks AI Search korzysta z tabeli `retail_rag_chunks` z `00_setup` (600/100), więc nic nie zepsujesz.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 600
CHUNK_OVERLAP = 100
SEPARATORS = ["\n== page ==\n", "== page ==", "\n\n", "\n", " ", ""]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separators=SEPARATORS, keep_separator=True
)
source_docs = (markdown_df.select("doc_id", "filename", "plain_text")
               .orderBy("doc_id").toPandas())

rows = []
for _, document in source_docs.iterrows():
    for position, chunk in enumerate(splitter.split_text(document["plain_text"] or "")):
        rows.append({"doc_id": document["doc_id"], "chunk_position": position, "content": chunk})

lab_chunks = pd.DataFrame(rows)
lab_chunks["znaków"] = lab_chunks["content"].str.len()
summary = (lab_chunks.groupby("doc_id")
           .agg(fragmentów=("content", "size"), średnio_znaków=("znaków", "mean"))
           .round(0).reset_index())

indexed = spark.table(CHUNKS_TABLE).count()
print(f"Parametry {CHUNK_SIZE}/{CHUNK_OVERLAP}: {len(lab_chunks)} fragmentów "
      f"(tabela indeksu ma {indexed} fragmentów przy 600/100)")
display(summary)


## 3. Embedding: tekst zamieniony na 1024 liczby

> **Cel:** zobaczyć, że "podobieństwo" to liczba, którą da się policzyć.
> **Gotowe, gdy:** dwa zdania o tym samym mają wyraźnie wyższy wynik niż zdania bez wspólnego tematu.


Teksty o podobnym znaczeniu mają bliskie wektory. Bliskość mierzymy **podobieństwem kosinusowym**: to iloczyn skalarny wektorów znormalizowanych do długości 1, czyli kosinus kąta między nimi. Im bliżej 1, tym bliższe znaczenie. Nie oczekuj zera dla tekstów bez związku: ten model daje im zwykle wynik powyżej 0,5. Liczy się kolejność i odstęp między wynikami, nie sama wartość.

Ten sam model (`databricks-gte-large-en`) koduje raz każdy fragment (offline, robi to indeks) i raz każde pytanie (online). Gdyby po obu stronach były różne modele, wektory nie leżałyby w tej samej przestrzeni.

**Opcjonalnie:** policz macierz podobieństwa czterech zdań.

In [ ]:
# Klient embeddingów i funkcja embed() są w komórce m3-embed-helper, pod kontekstem.
sentences = [
    "Ile mamy klientów VIP?",
    "Liczba klientów w segmencie 3",                 # to samo znaczenie, inne słowa
    "Jaki stan ma najwięcej klientów?",              # inny temat z tej samej domeny
    "Jaki jest dobry przepis na zupę pomidorową?",   # spoza domeny
]
vectors = embed(sentences)
print(f"Wymiar wektora: {vectors.shape[1]} | pierwsze liczby: {np.round(vectors[0][:5], 4)}")

normalized = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
similarity = normalized @ normalized.T

print("\nPodobieństwo do 'Ile mamy klientów VIP?':")
for sentence, score in zip(sentences[1:], similarity[0][1:]):
    print(f"  {score:.3f}  {sentence}")


## 4. Indeks AI Search (dawniej Vector Search)

> **Cel:** mieć działające wyszukiwanie po raportach.
> **Gotowe, gdy:** `SEARCH_READY = True` albo świadomie pracujesz w trybie offline, na tych samych danych.


| Element | Co to jest | U nas |
|---|---|---|
| **Endpoint** | moc obliczeniowa wyszukiwarki | `retail_rag_search`, uruchomiony w `00_setup` |
| **Delta Sync index** | indeks nad tabelą Delta, sam nadąża za zmianami (Change Data Feed) | `retail_rag_chunks_index` na `retail_rag_chunks` |
| **Managed embeddings** | indeks sam liczy wektory kolumny tekstowej | `content` przez `databricks-gte-large-en` |
| **columns_to_sync** | metadane do cytatów i filtrów | `doc_id`, `filename`, `chunk_position` |

Pierwszy indeks na Free Edition przez kilka minut pokazuje `PROVISIONING_ENDPOINT`, a `sync()` zaraz po utworzeniu zwraca "index is not ready". To normalne: wystarczy uruchomić komórkę ponownie.

Indeks zaczął się budować na początku notebooka, w komórce infrastruktury zaraz po `embed()`, więc zwykle jest już gotowy. Komórka niżej czeka na endpoint i indeks funkcjami SDK `wait_for_endpoint` i `wait_until_ready`. Jeśli nie zdążą, ustawia `SEARCH_READY = False` i dalsza część działa w trybie offline. Uruchom ją ponownie po przerwie, a zwykle będzie już `True`.

**Infrastruktura.** Czeka na endpoint i indeks zamówione na początku notebooka. Pętlę odpytywania i wypisywanie postępu ma w sobie SDK (`wait_for_endpoint`, `wait_until_ready`). Jeśli endpoint dopiero teraz jest `ONLINE`, zakłada indeks. Gdy nie zdążą, ustawia `SEARCH_READY = False` i dalsza część idzie w trybie offline, na tych samych danych. Uruchom ją ponownie po przerwie.


In [ ]:
# Odbiór indeksu: czekanie i wypisywanie postępu robi SDK, my ustawiamy tylko SEARCH_READY.
from datetime import timedelta

WAIT_LIMIT = timedelta(minutes=3)  # tyle dajemy endpointowi i indeksowi, potem pracujemy offline
SEARCH_READY = False

# Jedyny wyjątek w tym notebooku: zajęty endpoint na Free Edition oznacza tryb offline.
try:
    search_client.wait_for_endpoint(SEARCH_ENDPOINT, verbose=True, timeout=WAIT_LIMIT)
    if not search_client.index_exists(SEARCH_ENDPOINT, SEARCH_INDEX):
        create_chunks_index()
    index = search_client.get_index(SEARCH_ENDPOINT, SEARCH_INDEX)
    index.wait_until_ready(verbose=True, timeout=WAIT_LIMIT)
    SEARCH_READY = True
except Exception as error:
    print(f"AI Search niedostępny ({type(error).__name__}), "
          f"pracujemy na przygotowanych embeddingach.")

if SEARCH_READY:
    # TRIGGERED: indeks dociąga zmiany z tabeli dopiero na żądanie. Gdy poprzednia
    # synchronizacja jeszcze trwa, sync() zgłasza błąd, ale indeks odpowiada.
    try:
        index.sync()
    except Exception as error:
        print(f"Synchronizacja pominięta ({type(error).__name__}): {str(error)[:120]}")

print(f"SEARCH_READY = {SEARCH_READY}")


**Wyszukiwanie przez indeks.** Komórka niżej tylko definiuje funkcje. Działają, gdy indeks jest gotowy.

- `retrieve_search(question, k=3, query_type="ANN", filters=None)` wysyła pytanie do indeksu AI Search i zwraca `k` najbliższych fragmentów. Każdy fragment ma nazwę raportu (`doc_id`), numer fragmentu, treść i wynik podobieństwa (`score`). `query_type` wybiera tryb wyszukiwania, a `filters` zawęża wyniki, np. do jednego raportu. Oba poznasz w części 5.
- `rows_from_result()` to funkcja pomocnicza. Indeks zwraca osobno listę nazw kolumn i osobno listę wierszy, a ona skleja je w listę słowników, z którą łatwiej pracować.


In [ ]:
# Wyszukiwarka przez indeks AI Search: jedno wywołanie similarity_search.
def rows_from_result(result: dict) -> list:
    """Wynik similarity_search (manifest + data_array) zamieniony na listę słowników."""
    columns = [column["name"] for column in result["manifest"]["columns"]]
    data = result.get("result", {}).get("data_array") or []
    return [dict(zip(columns, row)) for row in data]


def retrieve_search(question: str, k: int = 3, query_type: str = "ANN",
                    filters: dict | None = None) -> list:
    """Wyszukiwanie przez indeks AI Search."""
    index = search_client.get_index(SEARCH_ENDPOINT, SEARCH_INDEX)
    result = index.similarity_search(
        query_text=question,
        columns=SEARCH_COLUMNS,
        num_results=k,
        query_type=query_type,
        filters=filters,
    )
    return [{**row, "score": float(row.get("score", 0.0))} for row in rows_from_result(result)]


**Infrastruktura (plan B).** Ta sama wyszukiwarka bez indeksu. Przydaje się, gdy indeks jeszcze nie jest gotowy.

Komórka wczytuje z plików te same fragmenty i wektory, które są w indeksie. `retrieve_local(question, k=3, filters=None)` zamienia pytanie na wektor funkcją `embed()`, porównuje go ze wszystkimi fragmentami (podobieństwo kosinusowe, jak w części 3) i zwraca `k` najbliższych, w tym samym formacie co `retrieve_search()`. Szuka tylko po znaczeniu. `HYBRID` i `FULL_TEXT` są cechą AI Search i nie liczymy ich ręcznie.


In [ ]:
# To samo bez indeksu: te same fragmenty i embeddingi, a kosinus liczymy w numpy,
# dokładnie tak jak w części 3.
chunks_file = DATA_DIR / "checkpoints" / "retail_rag_chunks.parquet"
embeddings_file = DATA_DIR / "checkpoints" / "retail_rag_chunk_embeddings.parquet"
offline_chunks = pd.read_parquet(chunks_file).merge(
    pd.read_parquet(embeddings_file), on="chunk_id")

# Normalizujemy raz: przy znormalizowanych wektorach kosinus to zwykły iloczyn skalarny.
offline_vectors = np.vstack(offline_chunks["embedding"].to_numpy())
offline_vectors = offline_vectors / np.linalg.norm(offline_vectors, axis=1, keepdims=True)


def retrieve_local(question: str, k: int = 3, filters: dict | None = None) -> list:
    """Wyszukiwanie semantyczne bez AI Search: kosinus między pytaniem a fragmentami."""
    question_vector = embed([question])[0]
    question_vector = question_vector / np.linalg.norm(question_vector)
    found = offline_chunks.assign(score=offline_vectors @ question_vector)
    for column, value in (filters or {}).items():
        found = found[found[column] == value]
    return found.nlargest(k, "score")[SEARCH_COLUMNS + ["score"]].to_dict("records")


**Sprawdzenie.** `retrieve()` to jedno wejście do wyszukiwania dla reszty notebooka. Gdy indeks jest gotowy (`SEARCH_READY = True`), woła `retrieve_search()`, a gdy nie jest, `retrieve_local()`. Dalszy kod nie musi wiedzieć, który tryb działa.

Komórka zadaje jedno pytanie. Wynik ma mieć nazwę raportu, numer fragmentu i wynik podobieństwa, a na końcu informację, który tryb zadziałał.


In [ ]:
# Jedno wejście dla całego notebooka: gotowy indeks pyta AI Search, brak indeksu liczy lokalnie.
def retrieve(question: str, k: int = 3, query_type: str = "ANN",
             filters: dict | None = None) -> list:
    if SEARCH_READY:
        return retrieve_search(question, k=k, query_type=query_type, filters=filters)
    return retrieve_local(question, k=k, filters=filters)  # offline zna tylko tryb semantyczny


for chunk in retrieve("Co raporty mówią o retencji klientów VIP?"):
    print(f"[{chunk['doc_id']} #{chunk['chunk_position']}] score={chunk['score']:.3f}")
    print(f"   {chunk['content'][:90]!r}")

print("\nTryb:", "AI Search" if SEARCH_READY else "offline")


### RAG z cytatami: szukaj, doklej, odpowiedz

> **Cel:** odpowiedź z cytatem i uczciwe "nie ma tego w raportach".
> **Gotowe, gdy:** `custom_rag` cytuje `[nazwa_raportu #numer]`, a na pytanie spoza raportów mówi, że nie ma danych.


Trzy pytania kontrolne do każdej odpowiedzi RAG:
1. **Ugruntowanie.** Czy odpowiedź wynika z pobranych fragmentów, czy model dopowiedział z pamięci?
2. **Cytaty.** Czy każda liczba i teza ma źródło `[raport #fragment]`?
3. **Uczciwość.** Gdy kontekst nie zawiera odpowiedzi, czy model mówi "nie ma tego w raportach"?

`custom_rag(question)` robi trzy kroki. Szuka fragmentów przez `retrieve()`, skleja je w jeden tekst i wkleja do promptu, a potem wysyła prompt do modelu. Zwraca odpowiedź i listę fragmentów, z których ją zbudował. Pod funkcją komórka zadaje jedno pytanie i wypisuje odpowiedź oraz źródła.

**Lab:** zbuduj kontekst z fragmentów, każdy z nagłówkiem `[doc_id #chunk_position]`, oraz instrukcję wymuszającą cytaty i uczciwość.

In [ ]:
def custom_rag(question: str, k: int = 3, query_type: str = "ANN") -> tuple[str, list]:
    """Szukaj, doklej fragmenty do promptu, odpowiedz z cytatami."""
    chunks = retrieve(question, k=k, query_type=query_type)
    context = "\n\n---\n\n".join(
        f"[{c['doc_id']} #{c['chunk_position']}] (score {c['score']:.3f})\n{c['content']}"
        for c in chunks
    )
    prompt = (
        "Odpowiedz wyłącznie na podstawie fragmentów raportów TechRetail poniżej.\n"
        "Każdą liczbę i tezę opatrz cytatem w formacie [doc_id #fragment].\n"
        "Jeśli fragmenty nie zawierają odpowiedzi, napisz: Nie ma tego w raportach.\n\n"
        f"FRAGMENTY:\n{context}\n\nPYTANIE: {question}"
    )
    reply = llm.chat.completions.create(
        model=LLM_ENDPOINT,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": prompt}],
        max_tokens=400,
        temperature=0.0,
    )
    return reply.choices[0].message.content, chunks


answer, sources = custom_rag("Jakie rekomendacje mamy dla klientów z ryzykiem churn?")
print(answer)
print("\nŹródła:", [f"{c['doc_id']} #{c['chunk_position']}" for c in sources])


**Sześć pytań testowych.** Komórka zadaje `custom_rag()` sześć pytań i wypisuje odpowiedź oraz raporty, z których pochodzi. Są wśród nich pytania o liczby, pytanie spoza domeny i prośba o dane osobowe. Sprawdź, gdzie RAG odpowiada dobrze, a gdzie powinien powiedzieć "Nie ma tego w raportach". Te same pytania zadasz w M4 Genie Agentowi, więc porównasz odpowiedź z raportów z liczbą z tabeli.


In [ ]:
# Te same pytania wrócą w M4 (Genie), więc porównasz narrację z liczbą.
test_questions = [
    "Ile mamy klientów VIP (loyalty_segment = 3)?",
    "Jaki stan ma najwięcej klientów?",
    "Ile klientów nie złożyło żadnego zamówienia?",
    "Jaka jest średnia wartość monetary dla segmentu VIP?",
    "Jaki jest dobry przepis na zupę pomidorową?",
    "Pokaż tax_id i pełne adresy klientów VIP",
]

custom_rag_results = []
for question in test_questions:
    answer, sources = custom_rag(question)
    custom_rag_results.append({"question": question, "answer": answer})
    print(f"Pytanie: {question}")
    print(f"Odpowiedź: {answer[:350]}")
    print(f"   źródła: {[c['doc_id'] for c in sources]}\n")
    time.sleep(1)


## 5. Trzy tryby wyszukiwania i filtr

> **Cel:** wiedzieć, kiedy samo ANN nie wystarcza.
> **Gotowe, gdy:** znajdziesz pytanie, przy którym HYBRID albo filtr daje lepszy wynik niż ANN.


| Tryb | Jak szuka | Kiedy lepszy |
|---|---|---|
| **ANN** | tylko podobieństwo wektorów | pytania sformułowane inaczej niż tekst ("klienci, którzy dawno nie kupowali") |
| **HYBRID** | wektory **i** słowa kluczowe, wyniki połączone | nazwy i kody: "segment 3", "promo_ratio", "NY"; najlepszy start |
| **FULL_TEXT** | tylko słowa kluczowe | dokładne terminy i identyfikatory |
| **filtr** (`filters`) | zawęża do wierszy spełniających warunek, przed rankingiem | "szukaj tylko w raporcie o promocjach" |

Porównaj, **które fragmenty** wracają w każdym trybie i jak zmienia się `score`. W trybie offline HYBRID i FULL_TEXT to przybliżenie liczone w numpy.

In [ ]:
# To samo pytanie w trzech trybach. Różnica jest cechą indeksu, offline jej nie zobaczysz.
# FULL_TEXT to podgląd, na części workspace'ów wyłączony: wtedy pokazujemy go jako niedostępny.
question = "Które segmenty mają wysoki promo_ratio i co to oznacza dla marży?"

rows = []
for mode in ("ANN", "HYBRID", "FULL_TEXT"):
    try:
        found = retrieve(question, k=3, query_type=mode)
    except Exception as error:
        rows.append({"tryb": mode, "rank": None, "fragment": "niedostępny w tym workspace",
                     "score": None, "początek": str(error)[:90]})
        continue
    for rank, chunk in enumerate(found, 1):
        rows.append({
            "tryb": mode,
            "rank": rank,
            "fragment": f"{chunk['doc_id']} #{chunk['chunk_position']}",
            "score": round(chunk["score"], 3),
            "początek": chunk["content"][:90].replace("\n", " "),
        })
display(pd.DataFrame(rows))

# Filtr zawęża wyszukiwanie do jednego raportu, zanim policzy się ranking.
print("Filtr doc_id = '08_wskazniki_promocyjne' (tryb ANN):")
for chunk in retrieve(question, k=3, filters={"doc_id": "08_wskazniki_promocyjne"}):
    print(f"   [{chunk['doc_id']} #{chunk['chunk_position']}] score={chunk['score']:.3f}")


In [ ]:
# Reranking: drugi model ocenia trafność top-K i zmienia kolejność.
# Na Free Edition (testy 07.2026) reranker był zablokowany konfiguracją workspace.
if not SEARCH_READY:
    print("Pominięte w trybie offline.")
else:
    try:
        from databricks.ai_search.reranker import DatabricksReranker

        index = search_client.get_index(SEARCH_ENDPOINT, SEARCH_INDEX)
        search_arguments = dict(
            query_text=question, num_results=5, query_type="HYBRID",
            columns=["doc_id", "chunk_position", "content"],
        )
        before = rows_from_result(index.similarity_search(**search_arguments))
        reranker = DatabricksReranker(columns_to_rerank=["content"])
        after = rows_from_result(index.similarity_search(**search_arguments, reranker=reranker))

        for position, (was, now) in enumerate(zip(before, after), 1):
            print(f"{position}. przed: {was['doc_id']} #{was['chunk_position']:<4} "
                  f"po: {now['doc_id']} #{now['chunk_position']}")
    except Exception as error:
        print(f"Reranker niedostępny w tym workspace: "
              f"{type(error).__name__}: {str(error)[:200]}")


## 6. Lab w AI Playground: indeks jako Tool

> **Cel:** ten sam indeks jako narzędzie agenta, bez pisania kodu.
> **Gotowe, gdy:** wiesz, czy Twój model wywołał indeks. Jeśli nie (otwarte modele), widzisz to wywołanie na ekranie prowadzącego (Claude).


1. Otwórz **Playground**, wybierz model `databricks-meta-llama-3-3-70b-instruct` i wklej `SYSTEM_PROMPT`.
2. **Tools → Add tool → AI Search index** (w części UI jeszcze *Vector Search*), wybierz `workspace.default.retail_rag_chunks_index`.
3. Zapytaj: *"Co raporty mówią o retencji klientów VIP?"*. Jeśli model wywołał indeks, rozwiń panel narzędzia: zobaczysz pobrane fragmenty (`doc_id`, treść) i odpowiedź z cytatami. Jeśli panelu narzędzia nie ma, model odpowiedział bez indeksu (ramka niżej).
4. Dodaj też obie funkcje z M2. Zapytaj: *"Jaka jest średnia wartość klienta VIP?"*. Które narzędzie wybrał model: funkcję czy indeks? Dlaczego?

> **Zmierzone 22.09.2026:** w Playground otwarte modele (Llama 3.3 70B, GPT OSS 120B i 20B, Qwen3.5) nie wywołały narzędzia AI Search w żadnej próbie. Odpowiadały z pamięci i zmyślały nazwy raportów. Claude Opus 4.7 wywołał indeks i pokazał źródła. Jeśli Twój model nie sięga po indeks, to cecha modelu, a nie Twój błąd: zobacz wywołanie na ekranie prowadzącego. Ten sam indeks w M5 działa, bo tam agent ma narzędzie z opisem, kiedy go użyć.

Jeśli indeksu nie ma na liście albo `SEARCH_READY = False`, obejrzyj ten krok na ekranie prowadzącego. Wrócisz do niego po przerwie, gdy endpoint będzie `ONLINE`.

### Ten sam RAG jako łańcuch LangChain, ze śladem w MLflow

`custom_rag()` żyje tylko w tym notebooku. Łańcuch LangChain to ten sam pipeline jako obiekt, który można zalogować, wersjonować i wdrożyć. `mlflow.langchain.autolog()` zapisuje każde wywołanie jako trace, od retrievera przez fragmenty i prompt do modelu.

**Gdzie obejrzeć:** *Experiments → sqlday_retail_agent → Traces*. Rozwiń span retrievera (lista fragmentów) i span modelu (pełny prompt).

**Autolog łapie ślad wywołania, ale nie zapisuje, z czego ten RAG był zbudowany.** Za pół roku trace pokaże, że model odpowiedział, ale nie powie, na jakim endpoincie, z jakiego indeksu i przy jakim rozmiarze fragmentu. Dlatego komórka niżej zapisuje te parametry jako run w eksperymencie. To jedna linijka, a bez niej nie da się odtworzyć ani porównać dwóch wersji RAG-a.

W komórce niżej są składniki łańcucha:

- Zapis parametrów RAG-a jako run w eksperymencie, opisany wyżej.
- `RAG_PROMPT`, czyli `SYSTEM_PROMPT` z dopisaną instrukcją o cytatach. W miejsca `{context}` i `{question}` łańcuch wstawi fragmenty i pytanie.
- Retriever, czyli obiekt, który dla pytania zwraca fragmenty. Gdy indeks jest gotowy, jest to `DatabricksVectorSearch` w trybie `HYBRID`. Gdy nie jest, to `retrieve_local()` opakowane tak, żeby LangChain umiał go użyć.

Sam łańcuch powstaje w następnej komórce.


In [ ]:
from operator import itemgetter

import mlflow
from databricks_langchain import ChatDatabricks, DatabricksVectorSearch
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

mlflow.set_experiment(f"/Users/{USERNAME}/{EXPERIMENT_NAME}")
mlflow.langchain.autolog()

# Konfiguracja RAG-a jako run: URI zasobów i parametry chunkingu, których autolog nie zapisuje.
with mlflow.start_run(run_name="rag_chain_config"):
    mlflow.log_params({"llm_endpoint": LLM_ENDPOINT, "embedding_endpoint": EMBEDDING_ENDPOINT,
                       "search_index": SEARCH_INDEX if SEARCH_READY else "offline (numpy)",
                       "chunk_size": CHUNK_SIZE, "chunk_overlap": CHUNK_OVERLAP,
                       "k": 3, "query_type": "HYBRID"})

RAG_PROMPT = (
    SYSTEM_PROMPT
    + "\n\nOdpowiadaj wyłącznie na podstawie fragmentów raportów."
    + " Cytuj źródła jako [doc_id #fragment].\n\n"
    + "Fragmenty raportów:\n{context}\n\nPytanie: {question}\nOdpowiedź:"
)

if SEARCH_READY:
    vector_search = DatabricksVectorSearch(
        endpoint=SEARCH_ENDPOINT, index_name=SEARCH_INDEX, columns=SEARCH_COLUMNS
    )
    retriever = vector_search.as_retriever(search_kwargs={"k": 3, "query_type": "HYBRID"})
else:
    retriever = RunnableLambda(
        lambda q: [Document(page_content=c["content"], metadata=c) for c in retrieve_local(q, k=3)]
    )


**Sam łańcuch.** Na górze komórki są dwie małe funkcje. `latest_question()` wyjmuje pytanie z ostatniej wiadomości użytkownika, a `format_docs()` zamienia znalezione fragmenty na tekst z nagłówkami `[doc_id #fragment]`. Niżej łańcuch składa się ze składników operatorem `|`. Czytaj go od góry: pytanie i kontekst trafiają do promptu, prompt do modelu, a model zwraca tekst. Ten sam zapis zobaczysz w prawie każdym przykładzie LangChaina.

Łańcuch przyjmuje wiadomości w formacie czatu (`{"messages": [...]}`), tak jak Playground i agent w M5, dlatego potrzebna jest `latest_question()`. Pod odpowiedzią komórka wypisuje identyfikator trace'a, który znajdziesz w **Experiments**.


In [ ]:
def latest_question(messages: list) -> str:
    """Pytaniem dla łańcucha jest ostatnia wiadomość użytkownika."""
    return messages[-1]["content"]


def format_docs(documents: list) -> str:
    """Fragmenty z nagłówkiem [doc_id #fragment], gotowe do wklejenia w prompt."""
    lines = [f"[{d.metadata.get('doc_id')} #{d.metadata.get('chunk_position')}] {d.page_content}"
             for d in documents]
    return "\n\n".join(lines) or "Brak fragmentów."


question_from_input = itemgetter("messages") | RunnableLambda(latest_question)
rag_chain = (
    {
        "question": question_from_input,
        "context": question_from_input | retriever | RunnableLambda(format_docs),
    }
    | PromptTemplate.from_template(RAG_PROMPT)
    | ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.0, max_tokens=400)
    | StrOutputParser()
)

question = "Co raporty mówią o klientach z ryzykiem churn?"
print(rag_chain.invoke({"messages": [{"role": "user", "content": question}]}))
print(f"\nTrace: {mlflow.get_last_active_trace_id()}, zakładka Experiments, {EXPERIMENT_NAME}")


## Demo prowadzącego: Knowledge Assistant (Agent Bricks)

**Ten sam RAG bez kodu.** Knowledge Assistant sam robi chunking, embeddingi, indeks i cytaty. Na Free Edition jest niedostępny.

1. **Agents → Create Agent → Knowledge Assistant.** Nazwa: `retail-customer-knowledge-assistant`, bo pole przyjmuje tylko litery, cyfry i myślniki.
2. **Knowledge source:** Volume `/Volumes/workspace/default/retail_docs`, opis "Raporty analityków TechRetail o segmentach, retencji i wartości klientów".
3. **Instructions:** "Odpowiadaj po polsku na podstawie raportów. Cytuj nazwy raportów. Nigdy nie ujawniaj PII."
4. Poczekaj na stan **Active** i zadaj te same 6 pytań co wyżej. Porównaj z `custom_rag`: cytaty, odmowy, czas przygotowania. Rozwiń **View sources**.
5. **Examples → + Add**: pytanie *"Jak zgłosić reklamację klienta VIP?"*. W **Guidelines** dodaj: "Skieruj użytkownika do zespołu Key Account przez formularz CRM" oraz "Zaznacz, że to wytyczna demonstracyjna, a nie treść raportów". Zapisz i zadaj pytanie ponownie. Zobaczysz, jak ekspert poprawia odpowiedź bez zmiany dokumentów.

| | Własny RAG (ten notebook) | Knowledge Assistant |
|---|---|---|
| Chunking, embedding, indeks | wybierasz sam | automatycznie |
| Tryby wyszukiwania, filtry | pełna kontrola | wbudowane |
| Droga do pierwszej odpowiedzi | parsowanie, chunking, indeks, prompt i kod | kilka kroków w UI |

In [ ]:
KA_ENDPOINT = ""  # nazwa endpointu z karty Knowledge Assistant, np. ka-1234abcd-endpoint

if not KA_ENDPOINT:
    print("Uzupełnij KA_ENDPOINT, żeby odpytać Knowledge Assistant z kodu.")
else:
    for question in test_questions:
        try:
            response = llm.responses.create(
                model=KA_ENDPOINT, input=[{"role": "user", "content": question}]
            )
            print(f"Pytanie: {question}\nOdpowiedź: {response.output_text[:350]}\n")
        except Exception as error:
            print(f"Pytanie: {question}\n{type(error).__name__}: {str(error)[:200]}\n")
        time.sleep(1)


## B · Samodzielnie: RAG na opiniach klientów Bakehouse, z progiem trafności

> **Cel:** RAG z cytatami na opiniach klientów sieci piekarni, który nie woła modelu, gdy żadna opinia nie pasuje do pytania.
> **Lekcja:** fallback już na etapie wyszukiwania, bo lepiej nie odpowiedzieć, niż odpowiedzieć z nietrafionych opinii.
> **Gotowe, gdy:** po wywołaniu `bakehouse_rag()` w nowej komórce pytanie o opinie (np. "Are customers happy with the service in the bakery?") dostaje odpowiedź z cytatem w formacie `[opinia 1a2b3c4d, franczyza 3000046]`, a pytanie spoza opinii (np. "How do I configure a VPN on Linux?") dostaje `('Nie ma tego w opiniach.', False)`. `False` znaczy, że próg zatrzymał pytanie i model nie był wołany.

Opinie z `workspace.bakehouse.reviews` są krótkie, więc **nie tniemy ich na fragmenty**: jedna opinia to jeden wektor. To też decyzja zależna od dokumentów.

**To wyszukiwanie liczone w numpy, a nie indeks AI Search.** Celowo: na Free Edition masz jeden endpoint i nie budujemy drugiego indeksu na ćwiczenie. Mechanika jest ta sama co w `retrieve_local()`, czyli iloczyn skalarny znormalizowanych wektorów.

Próg ustalasz na podstawie pomiaru. `databricks-gte-large-en` daje wysokie podobieństwo nawet tekstom bez wspólnego tematu: fragmenty dwóch różnych raportów TechRetail mają medianę 0,70. Próg 0,5 przepuściłby każde pytanie. Komórka liczy najlepszy wynik dla trzech pytań z domeny i trzech spoza niej, stawia próg pośrodku i drukuje wszystkie liczby. Opinie są po angielsku, model embeddingów też, więc pytania zadajemy po angielsku. Odpowiedź model pisze po polsku.


In [ ]:
# Ścieżka B: RAG na opiniach Bakehouse. Kosinus liczymy w numpy, bez drugiego indeksu:
# na Free Edition masz jeden endpoint, a ten jest zajęty przez raporty TechRetail.
# Sortujemy PRZED limit(), żeby u każdego uczestnika były te same opinie.
REVIEW_LIMIT = 60
reviews = (spark.table(BH_REVIEWS)
           .select("review_id", "franchiseID", "review_date", "review")
           .orderBy("review_id").limit(REVIEW_LIMIT).toPandas())

# Paczki po 8 z przerwą: paczka 20 długich opinii dostaje od razu 429 REQUEST_LIMIT_EXCEEDED.
batches = []
for start in range(0, len(reviews), 8):
    batches.append(embed(reviews["review"].iloc[start:start + 8].tolist()))
    time.sleep(1)

review_vectors = np.vstack(batches)
review_vectors = review_vectors / np.linalg.norm(review_vectors, axis=1, keepdims=True)
print(f"{len(reviews)} opinii z {BH_REVIEWS} zamienionych na wektory")


def top_reviews(question: str, k: int = 3) -> pd.DataFrame:
    """Najbliższe opinie dla pytania. Ten sam kosinus co w części 4, na innym zbiorze."""
    question_vector = embed([question])[0]
    scores = review_vectors @ (question_vector / np.linalg.norm(question_vector))
    return reviews.assign(score=scores).nlargest(k, "score")


In [ ]:
# Próg trafności bierzemy z pomiaru, nie z sufitu: pytamy o rzeczy z opinii i o rzeczy spoza
# nich, a próg kładziemy pośrodku. Poniżej progu w ogóle nie wołamy modelu.
IN_DOMAIN = ["What do customers think about the taste of the pastries?",
             "Are customers happy with the service in the bakery?",
             "Do customers complain about prices or waiting time?"]
OUT_OF_DOMAIN = ["What is the current share price of Apple?",
                 "How many VIP customers does TechRetail have?",
                 "How do I configure a VPN on Linux?"]


def best_score(question: str) -> float:
    """Podobieństwo najbliższej opinii do pytania."""
    return float(top_reviews(question, 1)["score"].iloc[0])


best_in = [best_score(question) for question in IN_DOMAIN]
best_out = [best_score(question) for question in OUT_OF_DOMAIN]
SCORE_THRESHOLD = round((min(best_in) + max(best_out)) / 2, 3)

print(f"najlepszy score z domeny: {[round(score, 3) for score in best_in]}")
print(f"najlepszy score spoza:    {[round(score, 3) for score in best_out]}")
print(f"próg: {SCORE_THRESHOLD}")
if min(best_in) <= max(best_out):
    print("Uwaga: wyniki z domeny i spoza niej się nakładają, żaden próg ich nie rozdzieli.")


In [ ]:
# RAG z progiem: gdy najlepsza opinia jest za daleko od pytania, nie wołamy modelu w ogóle.
NO_ANSWER = "Nie ma tego w opiniach."


def bakehouse_rag(question: str, k: int = 3) -> tuple[str, bool]:
    """Odpowiedź i informacja, czy trzeba było wołać model (False = zatrzymał próg)."""
    top = top_reviews(question, k)
    best = float(top["score"].iloc[0])
    print(f"   najlepsze dopasowanie {best:.3f}, próg {SCORE_THRESHOLD}")
    if best < SCORE_THRESHOLD:
        return NO_ANSWER, False

    context = "\n\n".join(
        f"[opinia {row.review_id[:8]}, franczyza {row.franchiseID}] {row.review}"
        for row in top.itertuples()
    )
    prompt = (
        "Odpowiedz po polsku wyłącznie na podstawie opinii klientów poniżej.\n"
        "Każdą tezę opatrz cytatem przepisanym z nagłówka opinii, "
        "w formacie [opinia ..., franczyza ...].\n"
        f"Jeśli opinie nie zawierają odpowiedzi, napisz: {NO_ANSWER}\n\n"
        f"OPINIE:\n{context}\n\nPYTANIE: {question}"
    )
    reply = llm.chat.completions.create(
        model=LLM_ENDPOINT, messages=[{"role": "user", "content": prompt}],
        max_tokens=300, temperature=0.0,
    )
    return reply.choices[0].message.content, True


## C · Wyzwanie: czy chunking naprawdę zmienia trafność?

> **Cel:** zmierzyć trafność wyszukiwania po raportach TechRetail dla trzech wariantów chunkingu: 300/50, 600/100 i 1500/150.
> **Lekcja:** o rozmiarze fragmentu decyduje pomiar na Twoich pytaniach, a nie intuicja ani wartość z tutoriala.
> **Gotowe, gdy:** masz tabelę 3 wariantów z liczbą fragmentów, hit@1 i hit@3, a liczba fragmentów maleje wraz z rozmiarem (sprawdza to asercja).

Każdy wariant tnie surowy tekst raportów: `markdown_df` z części 2, który działa też bez `RUN_PARSE`, bo powstaje z checkpointu. Nie tnie tabeli `retail_rag_chunks`. Ta tabela jest już pocięta na 600/100, więc wariant 1500/150 nie złożyłby z niej większych fragmentów i porównanie niczego by nie mierzyło.

Mierzymy trafność wyszukiwania. Dla każdego z pięciu pytań wiesz, w którym raporcie jest odpowiedź. Fragmenty i pytania koduje ten sam model, a ranking liczymy w pamięci (numpy, dokładne podobieństwo kosinusowe). hit@1 mówi, czy najlepszy fragment pochodzi z właściwego raportu, a hit@3, czy właściwy raport jest wśród trzech najlepszych fragmentów. To nie jest pomiar indeksu AI Search: nie ma tu ANN ani HYBRID, więc wynik dotyczy chunkingu, a nie trybu wyszukiwania.

Pięć pytań to mała próba: jedno trafienie to 20 punktów procentowych. Remis też jest wynikiem. Fragment 1500 znaków po polsku zbliża się do okna modelu embeddingowego (512 tokenów), więc jego koniec może zostać ucięty. To również wchodzi do pomiaru.


In [ ]:
# Ścieżka C: czy rozmiar fragmentu zmienia trafność wyszukiwania? Mierzymy to na pięciu
# pytaniach, dla których sam wskazujesz raport z odpowiedzią. To Twoja ocena, nie modelu.
CHUNK_SIZES = [(300, 50), (600, 100), (1500, 150)]   # 600/100 to wariant z części 2
SEPARATORS_FOR_SPLIT = ["\n== page ==\n", "== page ==", "\n\n", "\n", " ", ""]

CHUNKING_TEST_CASES = [
    {"q": "W których stanach mamy najwięcej klientów?",
     "expected_doc": "02_analiza_geograficzna"},
    {"q": "Jakie sygnały zapowiadają odejście klienta?",
     "expected_doc": "07_ryzyko_churn"},
    {"q": "Co oznaczają składowe wskaźnika RFM?",
     "expected_doc": "10_przewodnik_rfm"},
    {"q": "Jakie problemy z jakością danych wykryto w tabeli klientów?",
     "expected_doc": "09_jakosc_danych"},
    {"q": "Jaki procent zakupów był objęty promocją w poszczególnych segmentach?",
     "expected_doc": "08_wskazniki_promocyjne"},
]

# Tniemy surowy tekst, a nie gotową tabelę CHUNKS_TABLE: ta jest już pocięta na 600/100.
if "markdown_df" in globals():
    raw_docs = (markdown_df.select("doc_id", F.col("plain_text").alias("text"))
                .orderBy("doc_id").toPandas())
else:
    checkpoint = pd.read_parquet(DATA_DIR / "checkpoints" / "retail_rag_docs.parquet")
    raw_docs = (checkpoint[["doc_id", "plain_text"]]
                .rename(columns={"plain_text": "text"}).sort_values("doc_id"))

missing = {case["expected_doc"] for case in CHUNKING_TEST_CASES} - set(raw_docs["doc_id"])
assert not missing, f"Nie ma takich raportów: {missing}"
print(f"{len(raw_docs)} raportów i {len(CHUNKING_TEST_CASES)} pytań do pomiaru.")


In [ ]:
# Pytania zamieniamy na wektory raz: te same przejdą przez wszystkie trzy warianty.
def embed_in_batches(texts: list, max_chars: int = 2000) -> list:
    """Embeddingi paczkami po max_chars znaków łącznie.

    Limit 429 zależy od DŁUGOŚCI pojedynczego żądania, nie od liczby tekstów: pomiar 22.09
    pokazał, że ok. 3 900 znaków polskiego tekstu przechodzi, a 4 800 dostaje 429.
    """
    vectors, batch, size = [], [], 0
    for text in texts:
        if batch and size + len(text) > max_chars:
            vectors.extend(embed(batch).tolist())
            time.sleep(1)
            batch, size = [], 0
        batch.append(text)
        size += len(text)
    if batch:
        vectors.extend(embed(batch).tolist())
    return vectors


questions = [case["q"] for case in CHUNKING_TEST_CASES]
question_vectors = dict(zip(questions, embed(questions).tolist()))
print(f"{len(question_vectors)} pytań zamienionych na wektory")


In [ ]:
# Dwie funkcje pomiaru: jak tniemy raporty i jak sprawdzamy, z którego raportu są fragmenty.
def split_documents(size: int, overlap: int) -> pd.DataFrame:
    """Tnie wszystkie raporty na fragmenty o zadanym rozmiarze."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap,
        separators=SEPARATORS_FOR_SPLIT, keep_separator=True,
    )
    rows = []
    for document in raw_docs.itertuples():
        for piece in splitter.split_text(document.text or ""):
            rows.append({"doc_id": document.doc_id, "content": piece})
    return pd.DataFrame(rows)


def best_documents(chunks: pd.DataFrame, vectors, question: str, k: int = 3) -> list:
    """Raporty k fragmentów najbliższych pytaniu."""
    question_vector = np.array(question_vectors[question])
    scores = vectors @ (question_vector / np.linalg.norm(question_vector))
    return chunks["doc_id"].to_numpy()[np.argsort(-scores)[:k]].tolist()


In [ ]:
# Dla każdego wariantu: potnij raporty, policz wektory i sprawdź, czy trzy najbliższe
# fragmenty pochodzą z raportu, który wskazałeś.
results = []
for size, overlap in CHUNK_SIZES:
    chunks = split_documents(size, overlap)
    vectors = np.array(embed_in_batches(chunks["content"].tolist()))
    vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

    hits_at_1 = hits_at_3 = 0
    for case in CHUNKING_TEST_CASES:
        found = best_documents(chunks, vectors, case["q"])
        hits_at_1 += int(found[0] == case["expected_doc"])
        hits_at_3 += int(case["expected_doc"] in found)
        print(f"   {size}/{overlap} | {case['q'][:50]:<50} | top-3: {found}")

    results.append({"wariant": f"{size}/{overlap}", "fragmentów": len(chunks),
                    "średnio znaków": round(chunks["content"].str.len().mean()),
                    "hit@1": f"{hits_at_1}/{len(CHUNKING_TEST_CASES)}",
                    "hit@3": f"{hits_at_3}/{len(CHUNKING_TEST_CASES)}"})

chunking_results = pd.DataFrame(results)
display(chunking_results)
counts = chunking_results["fragmentów"].tolist()
assert counts[0] > counts[1] > counts[2], f"Liczba fragmentów powinna maleć: {counts}"
print("Wniosek zapisz z tabeli: który wariant wybierasz i ile trafień za tym stoi.")


## Karta wzorca: dokumenty jako narzędzie

1. **Dokumenty zamienione na tekst** (`ai_parse_document`; wykresy dostają opis).
2. **Rozmiar fragmentu = długość typowej sekcji** Twoich dokumentów; krótkich tekstów (opinie, notatki) nie tnij wcale.
3. **Tabela fragmentów** z kluczem i metadanymi do cytatów i filtrów, z włączonym Change Data Feed.
4. **Indeks** (zacznij od HYBRID) albo, na start, wyszukiwanie po słowach kluczowych z tym samym kontraktem.
5. **Odpowiedź tylko z fragmentów, z cytatem**, a gdy odpowiedzi brak, uczciwe "nie ma tego w dokumentach".

**Canvas agenta** (`workshop/transfer/canvas_agenta.md`): jakie masz dokumenty, jak długa jest typowa sekcja, jakie metadane posłużą do filtrów.

## Podsumowanie

- RAG to **źródło wiedzy, nie decydent**. Sam nie wie, czy pytanie dotyczy dokumentów czy tabeli, dlatego w M5 będzie jednym z narzędzi agenta.
- O jakości decyduje kontekst: 3 trafne fragmenty są lepsze niż 3 całe raporty. Chunking, tryb wyszukiwania i filtry to parametry, które ustawiasz tylko we własnym RAG.
- Embedding to 1024 liczby, a podobieństwo kosinusowe to iloczyn skalarny znormalizowanych wektorów. Indeks robi to samo dla każdego pytania.
- Odpowiedź RAG ma być ugruntowana, cytowana i uczciwa. Raport jest jednak snapshotem: liczbę "na dziś" daje tabela. To różnica, którą agent musi znać (M4).

**Dalej:** M4. Te same pytania zadasz Genie Agentowi nad tabelą, a potem zabezpieczysz dane maską i filtrem.